# Training ML Model for Underwriting - Google Colab

This notebook trains an XGBoost model for auto insurance underwriting using the research dataset.

## 1. Install Dependencies

In [ ]:
!pip install scikit-learn xgboost pandas numpy joblib

## 2. Upload the Dataset

Upload `labeled_research_data.json` from your local machine to Colab using the file upload button on the left.

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Load the dataset
with open('labeled_research_data.json', 'r') as f:
    data = json.load(f)

print(f"Loaded {len(data)} records")
print(f"\nFirst record sample:")
print(data[0])

## 3. Prepare Features and Labels

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Feature names
feature_names = [
    'age',
    'credit_score',
    'vehicle_value',
    'premium_amount',
    'claim_amount',
    'fraud_flag'
]

# Prepare features
X = []
for record in data:
    features = [
        record.get('age', 30),
        record.get('credit_score', 500),
        record.get('vehicle_value', 20000),
        record.get('premium_amount', 1000),
        record.get('claim_amount', 0),
        1 if record.get('fraud_flag', False) else 0
    ]
    X.append(features)

X = np.array(X)

# Prepare labels
y = [record.get('decision', 'Accept_Standard') for record in data]

# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y_encoded.shape}")
print(f"\nLabel classes: {label_encoder.classes_}")
print(f"Label distribution:")
for i, cls in enumerate(label_encoder.classes_):
    count = (y_encoded == i).sum()
    pct = count / len(y_encoded) * 100
    print(f"  {cls}: {count} ({pct:.1f}%)")

## 4. Split Data

In [ ]:
# Split into train/validation/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

## 5. Train XGBoost Model

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# Train XGBoost
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)

# Validate
y_val_pred = model.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_val, y_val_pred, target_names=label_encoder.classes_))

## 6. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

# Plot feature importance
importance = model.feature_importances_

plt.figure(figsize=(10, 6))
plt.barh(feature_names, importance)
plt.xlabel('Feature Importance')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.show()

print("Feature Importance:")
for name, imp in zip(feature_names, importance):
    print(f"  {name}: {imp:.4f}")

## 7. Evaluate on Test Set

In [ ]:
# Test set evaluation
y_test_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=label_encoder.classes_))

## 8. Save Model and Label Encoder

In [ ]:
import joblib
import pickle

# Save model
joblib.dump(model, 'underwriter_xgboost_model.pkl')
print("Model saved as underwriter_xgboost_model.pkl")

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print("Label encoder saved as label_encoder.pkl")

# Save feature names
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)
print("Feature names saved as feature_names.pkl")

## 9. Download the Files

Download the following files to your local machine:
- `underwriter_xgboost_model.pkl`
- `label_encoder.pkl`
- `feature_names.pkl`

Place them in your local project directory:
- `models/underwriter_xgboost_model.pkl`
- `models/label_encoder.pkl`
- `models/feature_names.pkl`

In [ ]:
# List files to download
import os

for file in ['underwriter_xgboost_model.pkl', 'label_encoder.pkl', 'feature_names.pkl']:
    if os.path.exists(file):
        size = os.path.getsize(file) / 1024  # KB
        print(f"{file}: {size:.2f} KB")
    else:
        print(f"{file}: NOT FOUND")